# Stage 2-2. Dataset and DataLoader

이 노트북은 `MnistDataset`과 `DataLoader`를 실습한다.

- `MnistDataset` — `load_mnist()` 위에서 정규화·task별 target 변환을 처리하는 Dataset 클래스
- `DataLoader` — `__len__`과 `__getitem__`을 구현한 Dataset을 받아 미니배치를 yield하는 범용 이터레이터

**학습 목표**
1. 3종 task별 `MnistDataset`의 images/targets shape과 dtype을 확인한다.
2. `MnistDataset.__getitem__(idx)`이 반환하는 단일 샘플 구조를 확인한다.
3. `DataLoader`를 통해 미니배치가 어떻게 구성되는지 확인한다.
4. task별 target 값 분포를 시각화한다.

## 0. 환경 설정

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt

from src.data.mnist import MnistDataset
from src.data.dataloader import DataLoader

## 1. MnistDataset — 3종 task 비교

`MnistDataset(split, task)`은 내부적으로 다음 순서로 처리한다.

1. `load_mnist(split)` → 원본 `(N, 28, 28) uint8` 이미지와 `(N,) uint8` 레이블 로드
2. `images.reshape(-1, 784).astype(float32) / 255.0` → 정규화
3. `transform_targets(labels, task)` → task별 target 배열 생성

In [ ]:
tasks = ["multiclass", "binary", "regression"]

datasets = {task: MnistDataset("train", task) for task in tasks}

print(f"{'task':<12} {'len':>6}  {'images.shape':<16} {'images.dtype':<10}  {'targets.shape':<16} {'targets.dtype'}")
print("-" * 80)
for task, ds in datasets.items():
    print(f"{task:<12} {len(ds):>6}  {str(ds.images.shape):<16} {str(ds.images.dtype):<10}  {str(ds.targets.shape):<16} {ds.targets.dtype}")

## 2. 단일 샘플 확인 (`__getitem__`)

`dataset[idx]`는 `(image, target)` tuple을 반환한다.

In [ ]:
idx = 0

for task, ds in datasets.items():
    image, target = ds[idx]
    print(f"[{task}]")
    print(f"  image  : shape={image.shape}, dtype={image.dtype}, min={image.min():.3f}, max={image.max():.3f}")
    print(f"  target : shape={target.shape}, dtype={target.dtype}, value={target}")
    print()

> **핵심**: 3종 task 모두 `image` shape은 동일하게 `(784,)`이다.  
> target은 multiclass만 `(10,)` one-hot이고, binary/regression은 `(1,)` 스칼라 배열이다.

## 3. task별 target 값 시각화

동일 인덱스의 이미지에 대해 3종 task의 target이 어떻게 다른지 비교한다.

In [ ]:
# 클래스 0~9 대표 샘플 인덱스 수집
ds_mc = datasets["multiclass"]
ds_bin = datasets["binary"]
ds_reg = datasets["regression"]

# multiclass targets에서 argmax로 레이블 복원 후 클래스별 첫 번째 인덱스 추출
class_indices = []
labels_restored = np.argmax(ds_mc.targets, axis=1)
for cls in range(10):
    idx = np.where(labels_restored == cls)[0][0]
    class_indices.append(idx)

fig, axes = plt.subplots(3, 10, figsize=(14, 5))
fig.suptitle("Task별 target 비교 (class 0~9 대표 샘플)", fontsize=12)

row_labels = ["multiclass\n(one-hot)", "binary\n(odd=1)", "regression\n(label/9)"]
task_datasets = [ds_mc, ds_bin, ds_reg]

for row, (task_ds, row_label) in enumerate(zip(task_datasets, row_labels)):
    for col, idx in enumerate(class_indices):
        image, target = task_ds[idx]
        ax = axes[row, col]
        ax.imshow(image.reshape(28, 28), cmap="gray")
        ax.axis("off")
        if row == 0:
            ax.set_title(f"cls {col}", fontsize=8)
        # target 값 표시
        if row == 0:
            val_str = str(int(np.argmax(target)))
        elif row == 1:
            val_str = str(int(target[0]))
        else:
            val_str = f"{target[0]:.2f}"
        ax.set_xlabel(val_str, fontsize=7)

    axes[row, 0].set_ylabel(row_label, fontsize=8, rotation=0, labelpad=55, va="center")

plt.tight_layout()
plt.show()

## 4. DataLoader — 미니배치 구조 확인

`DataLoader(dataset, batch_size, shuffle)`는 `__iter__`에서 `(images_batch, targets_batch)` tuple을 yield한다.

In [ ]:
batch_size = 64

for task, ds in datasets.items():
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    images_batch, targets_batch = next(iter(loader))
    print(f"[{task}]")
    print(f"  len(loader)     : {len(loader)} batches")
    print(f"  images_batch    : shape={images_batch.shape}, dtype={images_batch.dtype}")
    print(f"  targets_batch   : shape={targets_batch.shape}, dtype={targets_batch.dtype}")
    print()

## 5. shuffle 동작 확인

`shuffle=True`일 때 매 epoch마다 배치 순서가 달라지는 것을 확인한다.

In [ ]:
ds = datasets["multiclass"]
loader_no_shuffle = DataLoader(ds, batch_size=8, shuffle=False)
loader_shuffle    = DataLoader(ds, batch_size=8, shuffle=True)

# 첫 배치의 이미지 첫 픽셀값으로 순서 비교
imgs_no, _ = next(iter(loader_no_shuffle))
imgs_sh, _ = next(iter(loader_shuffle))

print("shuffle=False, 첫 배치 첫 픽셀:", imgs_no[:, 0])
print("shuffle=True,  첫 배치 첫 픽셀:", imgs_sh[:, 0])
print()
print("→ shuffle=True일 때 값이 다름:", not np.allclose(imgs_no[:, 0], imgs_sh[:, 0]))

## 6. 전체 이터레이션 확인

`DataLoader` 전체를 순회하며 배치 수와 마지막 배치 크기를 검증한다.

In [ ]:
ds = datasets["multiclass"]
loader = DataLoader(ds, batch_size=64, shuffle=False)

batch_sizes = []
for images_b, targets_b in loader:
    batch_sizes.append(len(images_b))

print(f"총 배치 수     : {len(batch_sizes)}")
print(f"일반 배치 크기 : {batch_sizes[0]}")
print(f"마지막 배치 크기: {batch_sizes[-1]}  ← {len(ds)} % 64 = {len(ds) % 64}")
print(f"전체 샘플 합계 : {sum(batch_sizes)} (== len(dataset): {sum(batch_sizes) == len(ds)})")

## 7. 정리

| 항목 | 내용 |
|---|---|
| `MnistDataset.images` | `(N, 784)` float32, 범위 0.0~1.0 |
| `MnistDataset.targets` (multiclass) | `(N, 10)` float32, one-hot |
| `MnistDataset.targets` (binary) | `(N, 1)` float32, 0 or 1 |
| `MnistDataset.targets` (regression) | `(N, 1)` float32, 0.0~1.0 |
| `DataLoader.__len__()` | `ceil(N / batch_size)` |
| `DataLoader.__iter__()` | `(images_batch, targets_batch)` tuple yield |

**핵심 설계 원칙**
- `MnistDataset`은 정규화와 task 변환을 한 번에 처리하므로, 클라이언트는 task만 지정하면 된다.
- `DataLoader`는 `__len__`과 `__getitem__`을 구현한 Dataset이면 종류에 관계없이 수용한다.
- 마지막 배치는 `N % batch_size`개로 drop 없이 반환된다.